In [2]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"     # use PyTorch's sampler, no compilation needed
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # engine runs inside the notebook process

In [3]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [11]:
import sys; print(sys.executable)

/home/user/miniconda/bin/python


In [12]:
from huggingface_hub import HfApi, whoami

try:
    print("Logged in as:", whoami()["name"])
except Exception as e:
    print("NOT logged in:", e)

print([d.id for d in HfApi().list_datasets(author="Maci10")])

Logged in as: Maci10
['Maci10/dpo-king-sft_all', 'Maci10/dpo-king-math_val', 'Maci10/dpo-king-math500']


In [6]:
from huggingface_hub import login
login()   # a box appears; paste the token there (never in the code itself)

In [ ]:
math500.push_to_hub("Maci10/dpo-king-math500", split="train", private=True)

In [13]:
from datasets import load_dataset
math500 = load_dataset('Maci10/dpo-king-math500', split = 'test')
print(len(math500))

500


In [14]:
sft_all = load_dataset('Maci10/dpo-king-sft_all', split = 'train')
math_val = load_dataset('Maci10/dpo-king-math_val', split = 'train')

print(len(sft_all), len(math_val))

122416 502


In [4]:
import sys; sys.path.insert(0, ".")
from vllm import LLM, SamplingParams
from posttrain.prompts import build_prompt
from posttrain.grader import mv_grade
from posttrain.data import load_split
from datasets import load_dataset

math500 = load_dataset('Maci10/dpo-king-math500', split = 'test')

llm = LLM(
    model = "Qwen/Qwen3-0.6B-Base",
    dtype = "bfloat16",
    max_model_len = 2048,
    gpu_memory_utilization = 0.85,
    seed = 0
)


# Stop on token IDs: <|im_end|> is a special token, and vLLM strips special
# tokens from the output text, so a stop *string* can miss it.
tok = llm.get_tokenizer()
stop_ids = [tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")]
params = SamplingParams(temperature=0.0, max_tokens = 1024, stop_token_ids = stop_ids,)

batch = math500.select(range(5))
prompts = [build_prompt(ex['problem']) for ex in batch]
outputs = llm.generate(prompts, params)

for ex, out in zip(batch, outputs):
    text = out.outputs[0].text
    print("GOLD:", ex["answer"], "| CORRECT:", mv_grade(ex["answer"], text),
          "| tokens:", len(out.outputs[0].token_ids))
    print(text[-300:], "\n" + "-" * 60)

W1115 03:26:48.656000 12734 site-packages/torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1115 03:26:48.658000 12734 site-packages/torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1115 03:26:48.659000 12734 site-packages/torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=12734) INFO 11-15 03:26:54 [startup.py:96] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-0.6B-Base'}
(MainProcess pid=12734) WARNING 11-15 03:26:54 [arg_utils.py:1984] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=12734) INFO 11-15 03:26:54 [model.py:744] Resolved architecture: Qwen3ForCausalLM
(MainProcess pid=12734) INFO 11-15 03:26:54 [model.py:2170] Using max model len 2048
(MainProcess pid=12734) INFO 11-15 03:26:54 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=12734) INFO 11-15 03:26:55 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(MainProcess pid=12734) INFO 11-15 03:26:57 [core.py:139] Initializing a V1 LLM engine (v0.33.0) with config: model='Qwen/Qwen3-0.6B-Base', speculative_config=None, tokenizer='Qwen/Qwen3-0.6B-Base', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


... (4 outputs left out)

(MainProcess pid=12734) INFO 11-15 03:27:13 [model_runner.py:1163] Graph capturing finished in 5 secs, took 0.18 GiB
(MainProcess pid=12734) INFO 11-15 03:27:13 [gpu_worker.py:1003] CUDA graph pool memory: 0.18 GiB (actual), 0.32 GiB (estimated), difference: 0.13 GiB (74.2%).
(MainProcess pid=12734) INFO 11-15 03:27:13 [gpu_worker.py:1066] Free memory on device (21.78/22.03 GiB) on startup. Desired GPU memory utilization is (0.85, 18.73 GiB). Actual usage is 1.38 GiB for consumed memory (weights + non-torch), 0.56 GiB for peak activation, and 0.18 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=17676745728` (16.46 GiB) to fit into requested memory, or `--kv-cache-memory=20952988672` (19.51 GiB) to fully utilize gpu memory. Current kv cache memory in use is 16.79 GiB.
(MainProcess pid=12734) INFO 11-15 03:27:14 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(MainProcess pid=12734) 

Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 5/5 [00:07<00:00,  1.41s/it, est. speed input: 98.79 toks/s, output: 552.22 toks/s]

GOLD: \left( 3, \frac{\pi}{2} \right) | CORRECT: False | tokens: 1024
 ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ ⚇ 
------------------------------------------------------------
GOLD: p - q | CORRECT: False | tokens: 1024
te sums, one over \(j\) and one over \(k\):

\[
\sum_{j=1}^\infty \sum_{k=1}^\infty \frac{1}{(j+k)^3} = \sum_{j=1}^\infty \sum_{k=1}^\infty \frac{1}{(j+k)^3} = \sum_{j=1}^\infty \sum_{k=1}^\infty \frac{1}{(j+k)^3}.
\]

We can also split this sum into two separate sums, one over \(j\) and one over \( 
------------------------------------------------------------
GOLD: \frac{14}{3} | CORRECT: True | tokens: 436
ns, we need a common denominator. The common denominator for 1 and \(\frac{5}{3}\) is 3:
\[
2 = \frac{6}{3}
\]
